# Risk/UQ Paper Track: Risk Model Training

## Objective
Train and calibrate the CRAR risk model with persistent artifacts so long runs survive Colab interruptions and can resume safely.

## Why This Notebook Exists
This notebook is the only place where expensive candidate dataset generation and model fitting happen. The benchmark and paper notebooks should reuse these persisted outputs.


## Hypotheses Being Tested
- **H1:** Post-hoc temperature scaling improves reliability (ECE/NLL) vs raw probabilities.
- **H2:** A compact ensemble captures useful epistemic uncertainty for safety reranking.
- **H3:** Calibrated `failure_proxy_h15` can support a meaningful risk budget (`<= 0.20`).


## Methodology
1. Bootstrap deterministic Colab environment and mount Drive.
2. Resolve persistent run identity (`RUN_TAG`, `PERSIST_ROOT`, `RUN_MODE`).
3. Load persisted dataset/model artifacts when available (`auto`/`resume`).
4. Otherwise build candidate dataset, train ensemble, calibrate heads, and write artifacts.

`RUN_MODE` semantics:
- `auto`: reuse existing artifacts when found, else compute.
- `resume`: require existing artifacts, fail if missing.
- `fresh`: ignore existing artifacts and recompute.


## Step 1 - Bootstrap Environment (Repo + Drive + Runtime)
Use the same runtime bootstrap pattern as the closed-loop simulation notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Configure Persistent Run Identity
This determines where all training artifacts are stored on Drive.


In [ ]:
from src.workflows import initialize_run_context, report_run_context

# Leave RUN_TAG empty to auto-adopt latest matching run under PERSIST_ROOT.
RUN_TAG = ''
RUN_TAG_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'
RUN_MODE = 'auto'  # auto | fresh | resume

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=1,
    shard_id=0,
    auto_run_main_loop_when_ready=False,
    run_main_loop_override=False,
    run_tag_prefix=RUN_TAG_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=True,
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
run_prefix = run_context.run_prefix
print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)


## Step 3 - Configure Risk/UQ Training Knobs

In [ ]:
# Risk/UQ experiment knobs
cfg.risk_dataset_build = True
cfg.risk_dataset_candidate_count = 8
cfg.risk_dataset_control_horizon_steps = 6
cfg.risk_dataset_label_horizons = (5, 10, 15)
cfg.risk_model_ensemble_size = 5
cfg.risk_model_hidden_dims = (128, 128)
cfg.risk_model_dropout = 0.10
cfg.risk_model_learning_rate = 1e-3
cfg.risk_model_batch_size = 1024
cfg.risk_model_max_epochs = 50
cfg.risk_model_patience = 8

# Epoch-level persistence for interruption tolerance.
cfg.risk_model_checkpoint_every_epochs = 1
cfg.risk_model_resume_from_checkpoints = True

cfg.risk_calibration_method = 'temperature'
cfg.risk_conformal_alpha = 0.10
cfg.risk_control_fail_budget = 0.20
cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg.uq_eval_probability_bins = 15
cfg


## Step 4 - Execute Resume-Aware Training Flow
This cell is interruption-safe in two layers:
- artifact-level: reuses finished dataset/model outputs in `auto/resume` mode;
- epoch-level: if training was interrupted mid-epoch-loop, it resumes from per-member checkpoints.


In [ ]:
import pandas as pd
from src.closedloop.core import build_closedloop_runner_and_splits
from src.workflows import (
    has_existing_risk_model_artifacts,
    has_existing_risk_training_checkpoints,
    load_existing_risk_dataset_artifact,
    run_risk_training_flow,
)

existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
existing_model_ready = has_existing_risk_model_artifacts(cfg.run_prefix)
existing_checkpoint_ready = has_existing_risk_training_checkpoints(cfg.run_prefix)

if (RUN_MODE == 'fresh'):
    existing_dataset_df = pd.DataFrame()

print('existing_model_ready =', existing_model_ready)
print('existing_checkpoint_ready =', existing_checkpoint_ready)

if existing_dataset_df.empty:
    print('[risk-train] no persisted dataset found; building simulation runner and dataset...')
    runner, data, reference_idx, candidate_eval_idx, eval_idx_all, eval_idx, reference_df, base_eval_openloop_df = build_closedloop_runner_and_splits(
        cfg=cfg,
        data_iter=None,
        dataset_config=None,
        n_shards=1,
        shard_id=0,
    )
    bundle = run_risk_training_flow(
        cfg=cfg,
        runner=runner,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )
else:
    print(f'[risk-train] using persisted dataset rows={len(existing_dataset_df)}')
    bundle = run_risk_training_flow(
        cfg=cfg,
        dataset_df=existing_dataset_df,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )

print('loaded_from_existing =', bundle.loaded_from_existing)
print('dataset_rows =', len(bundle.dataset_bundle.dataset_df))
print('artifact_keys =', sorted(bundle.artifact_paths.keys()))


## Report-Style Notes (Fill After Run)
### Dataset
- rows/scenarios processed:
- train/val/test/holdout composition:

### Training
- validation NLL/Brier before calibration:
- validation NLL/Brier after calibration:
- learned temperature distribution across heads:

### Decision Readiness
- conformal threshold (`failure_proxy_h15`):
- expected effect on control reranking:


In [ ]:
# Optional: inspect validation calibration table
# bundle.calibration_bundle.summary_df
